# Phase M2-A1 — nnUNet v2 Fine-Tuning (MU-Glioma-Post)## 3 output channels: WT / TC / ET (Same label convention as BraTS 2024)**Starting weights**: BraTS 2021 fold_0 pretrained (original — NOT BraTS 2024 fine-tuned)**Data**: 596 MU-Glioma-Post scans (203 patients, up to 6 timepoints each)**Architecture**: PlainConvUNet, 30.8M params, 6 stages, features [32,64,128,256,320,320]### Kaggle Datasets Required:1. `mu-glioma-post` — MU-Glioma NIfTI imaging data2. `nnunet-brats2021-fold0` — nnUNet v2 pretrained weights (`checkpoint_final.pth` + `plans.json`)3. `mu-glioma-m1-outputs` — M1 pipeline outputs (`scan_index.json`, `data_splits.json`, `mu_glioma_master.csv`)### Pipeline:1. Load BraTS 2021 pretrained nnUNet2. Fine-tune on MU-Glioma (30 epochs, Dice+CE loss)3. Extract embeddings: octant(8C) + region(3C) + vol(9) = 2825-D per scan4. Save embeddings + spatial tokens + treatment tokens for downstream TaViT V3

In [ ]:
import re
import math
import time
import json
import random
import shutil
import subprocess
import numpy as np
import torch
import torch.nn.functional as F
import gc
from pathlib import Path
import os, warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = 'nnunet'
PATCH       = [128, 128, 128]
# 3-channel output matching BraTS 2021 pretrained checkpoint (no RC)
# Output order: WT, TC, ET  (same region-based sigmoid as nnUNet)
REGIONS     = ['WT', 'TC', 'ET']
OUTPUT_ROOT = Path('/kaggle/working/phase_m2_nnunet')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f'Model: {MODEL_NAME} | Patch: {PATCH} | Regions: {REGIONS}')

In [ ]:
import subprocess, sys, json, time, math, os, shutil
import numpy as np
import torch
import torch.nn.functional as F

# Install nnunetv2 (needed to reconstruct exact architecture from plans)
try:
    import nnunetv2
    try:
        ver = nnunetv2.__version__
    except AttributeError:
        import importlib.metadata
        try: ver = importlib.metadata.version('nnunetv2')
        except Exception: ver = 'installed'
    print(f'nnunetv2 {ver} ready')
except ImportError:
    print('Installing nnunetv2 ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nnunetv2', '-q'])
    import nnunetv2
    print('nnunetv2 installed')

try:
    import monai
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'monai[all]', '-q'])
    import monai

import monai.transforms as T
from monai.data import Dataset, CacheDataset, DataLoader
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.utils import set_determinism
from monai.transforms import MapTransform

set_determinism(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'MONAI {monai.__version__} | PyTorch {torch.__version__} | Device: {device}')
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | Total memory: {total_mem:.1f} GB')

# Disk space check
usage = shutil.disk_usage('/kaggle/working')
free_gb = usage.free / 1e9
print(f'Disk free: {free_gb:.1f} GB (need ~2GB for checkpoints)')
if free_gb < 5:
    print('WARNING: low disk space!')

In [ ]:
# MU-Glioma-Post ground truth labels (IDENTICAL to BraTS 2024):
#   0 = Background
#   1 = NETC (Non-Enhancing Tumor Core — necrosis/cysts)
#   2 = SNFH (Surrounding Non-enhancing FLAIR Hyperintensity — edema)
#   3 = ET   (Enhancing Tissue — active tumor)
#   4 = RC   (Resection Cavity — fluid/blood/air)
#
# Evaluation sub-regions (same as BraTS 2024):
#   ET = label 3
#   TC = ET + NETC       = labels 3 + 1
#   WT = ET + SNFH + NETC = labels 3 + 2 + 1
#   RC is NOT part of WT or TC!
#
# 3-channel output: [WT, TC, ET]

class ConvertToMultiChannelBrats3Chd(MapTransform):
    def __call__(self, data):
        d = dict(data)
        for key in self.key_iterator(d):
            img = d[key]
            if img.ndim == 4 and img.shape[0] == 1:
                img = img.squeeze(0)
            result = [
                (img==1)|(img==2)|(img==3),  # WT = NETC+SNFH+ET (no RC)
                (img==1)|(img==3),           # TC = NETC+ET
                img==3,                      # ET = Enhancing Tissue
            ]
            d[key] = (torch.stack(result, 0).float()
                      if isinstance(img, torch.Tensor)
                      else np.stack(result, 0).astype(np.float32))
        return d

print('MU-Glioma Label mapping: WT=1+2+3 | TC=1+3 | ET=3')
print('  1=NETC  2=SNFH  3=ET  4=RC(excluded)')

In [ ]:
# MU-Glioma Data Discovery
# Kaggle renames .nii.gz to .nii_gz -- symlink trick restores proper extension
import nibabel as nib
import os

SYMLINK_DIR = Path('/kaggle/working/nifti_links')

def setup_nii_gz_symlinks(data_dir):
    count = 0
    for nii_gz in Path(data_dir).rglob('*.nii_gz'):
        rel = nii_gz.relative_to(data_dir)
        real_name = nii_gz.name.replace('.nii_gz', '.nii.gz')
        link = SYMLINK_DIR / rel.parent / real_name
        link.parent.mkdir(parents=True, exist_ok=True)
        if not link.exists():
            os.symlink(str(nii_gz), str(link))
            count += 1
    return count

DATA_ROOT = Path('/kaggle/input')
for ds_dir in DATA_ROOT.iterdir():
    if not ds_dir.is_dir(): continue
    nii_gz_files = list(ds_dir.rglob('*.nii_gz'))
    if nii_gz_files:
        n = setup_nii_gz_symlinks(ds_dir)
        if n: print(f'  Created {n} symlinks from {ds_dir.name}')

scan_index_path = None
splits_path = None
master_csv_path = None
for p in DATA_ROOT.rglob('scan_index.json'):
    scan_index_path = p; break
for p in DATA_ROOT.rglob('data_splits.json'):
    splits_path = p; break
for p in DATA_ROOT.rglob('mu_glioma_master.csv'):
    master_csv_path = p; break

print(f'scan_index.json: {scan_index_path}')
print(f'data_splits.json: {splits_path}')
print(f'master_csv.csv: {master_csv_path}')

if scan_index_path is None:
    raise RuntimeError('scan_index.json not found')

with open(scan_index_path) as f:
    scan_index = json.load(f)
scans_dict = scan_index['scans']
print(f'Total scans in index: {len(scans_dict)}')

if splits_path:
    with open(splits_path) as f:
        splits = json.load(f)
    train_pids = set(splits['train'])
    val_pids   = set(splits['val'])
    test_pids  = set(splits['test'])
    print(f'Splits: Train={len(train_pids)} | Val={len(val_pids)} | Test={len(test_pids)}')
else:
    all_pids = sorted(set(s['patient_id'] for s in scans_dict.values()))
    n80 = int(0.8 * len(all_pids))
    train_pids = set(all_pids[:n80])
    val_pids = set(all_pids[n80:])
    test_pids = set()
    print(f'No splits file, 80/20: Train {len(train_pids)} | Val {len(val_pids)}')

NIFTI_ROOT = None
for search_root in [SYMLINK_DIR, DATA_ROOT]:
    if not search_root.exists(): continue
    for p in search_root.rglob('PatientID_0003'):
        if p.is_dir():
            NIFTI_ROOT = p.parent
            break
    if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    for search_root in [SYMLINK_DIR, DATA_ROOT]:
        if not search_root.exists(): continue
        for pat in ['*brain_t1c.nii.gz', '*brain_t1c.nii_gz', '*brain_t1c.nii']:
            for p in search_root.rglob(pat):
                NIFTI_ROOT = p.parent.parent.parent
                break
            if NIFTI_ROOT: break
        if NIFTI_ROOT: break

if NIFTI_ROOT is None:
    raise RuntimeError('Could not find MU-Glioma NIfTI data')
print(f'NIFTI_ROOT: {NIFTI_ROOT}')

all_scans = []
for scan_id, scan_data in scans_dict.items():
    pid = scan_data['patient_id']
    tp  = scan_data['timepoint']
    tp_dir = NIFTI_ROOT / pid / f'Timepoint_{tp}'
    paths = {}
    for mod in ['t1c', 't1n', 't2f', 't2w']:
        candidates = list(tp_dir.glob(f'*brain_{mod}*')) if tp_dir.exists() else []
        if candidates:
            paths[mod] = str(candidates[0])
    mask_candidates = list(tp_dir.glob('*tumorMask*')) if tp_dir.exists() else []
    if mask_candidates:
        paths['seg'] = str(mask_candidates[0])
    if not all(m in paths for m in ['t1c', 't1n', 't2f', 't2w', 'seg']):
        continue
    entry = {
        'scan_id': scan_id, 'patient_id': pid, 'timepoint': str(tp),
        't1n': paths['t1n'], 't1c': paths['t1c'],
        't2w': paths['t2w'], 't2f': paths['t2f'],
        'seg': paths['seg'],
        'split': 'train' if pid in train_pids else ('val' if pid in val_pids else 'test'),
    }
    all_scans.append(entry)

train_scans = [s for s in all_scans if s['split'] == 'train']
val_scans   = [s for s in all_scans if s['split'] == 'val']
print(f'Resolved: {len(all_scans)} total scans')
print(f'Train: {len(train_scans)} | Val: {len(val_scans)}')

if all_scans:
    test_path = all_scans[0]['t1c']
    print(f'Spot-check: {test_path}')
    print(f'  exists={Path(test_path).exists()}, size={Path(test_path).stat().st_size}')
    try:
        img = nib.load(test_path)
        print(f'  nibabel OK: shape={img.shape}')
    except Exception as e:
        print(f'  nibabel FAIL: {e}')

In [ ]:
patch = [128, 128, 128]
train_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats3Chd(keys=['label']),
    T.RandFlipd(keys=['image','label'], spatial_axis=[0], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[1], prob=0.5),
    T.RandFlipd(keys=['image','label'], spatial_axis=[2], prob=0.5),
    T.RandScaleIntensityd(keys='image', factors=0.1, prob=0.3),
    T.RandShiftIntensityd(keys='image', offsets=0.1, prob=0.3),
    T.SpatialPadd(keys=['image','label'], spatial_size=patch),
    T.RandCropByPosNegLabeld(keys=['image','label'], label_key='label',
        spatial_size=patch, pos=2, neg=1, num_samples=2, image_key='image', image_threshold=0),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
val_transforms = T.Compose([
    T.LoadImaged(keys=['image','label']),
    T.EnsureChannelFirstd(keys=['image','label']),
    T.EnsureTyped(keys=['image','label']),
    T.Orientationd(keys=['image','label'], axcodes='RAS'),
    T.CropForegroundd(keys=['image','label'], source_key='image', allow_smaller=True),
    T.NormalizeIntensityd(keys='image', nonzero=True, channel_wise=True),
    ConvertToMultiChannelBrats3Chd(keys=['label']),
    T.EnsureTyped(keys=['image','label'], dtype=torch.float32),
])
print('Transforms ready (3-channel: WT/TC/ET)')

In [ ]:
# Validation: existence + size check (symlinks fix .nii_gz extension)
def validate_scan(s):
    try:
        for key in ['t1n','t1c','t2w','t2f','seg']:
            p = Path(s[key])
            if not p.exists(): return False
            if p.stat().st_size < 1024: return False
        return True
    except Exception:
        return False

def build_dicts(scan_list, label=''):
    dicts, bad = [], []
    for s in scan_list:
        if not validate_scan(s):
            bad.append(s['scan_id']); continue
        dicts.append({
            'image': [s['t1n'],s['t1c'],s['t2w'],s['t2f']],
            'label': s['seg'],
            'patient_id': s['patient_id'],
            'timepoint':  s['timepoint'],
        })
    if bad: print(f'  {label}Skipped {len(bad)} missing: {bad[:3]}{"..." if len(bad)>3 else ""}')
    return dicts

print('Validating scans...')
train_dicts = build_dicts(train_scans, 'Train: ')
val_dicts   = build_dicts(val_scans, 'Val: ')
all_dicts   = build_dicts(all_scans, 'All: ')
print(f'Train dicts: {len(train_dicts)} | Val dicts: {len(val_dicts)} | All dicts: {len(all_dicts)}')
if train_dicts:
    print(f'  Sample: {train_dicts[0]["image"][0]}')
    print(f'  Exists: {Path(train_dicts[0]["image"][0]).exists()}')

In [ ]:
print('='*55)
print('  Loading nnUNet v2 pretrained model')
print('  Source: BraTS 2021 fold_0 (ORIGINAL — not BraTS 2024 fine-tuned)')
print('  Target: WT=0.9005  TC=0.8673  ET=0.8509 (BraTS 2021)')
print('='*55)

# ── Step 1: Find checkpoint + plans ──
ckpt_path  = None
plans_path = None
for p in Path('/kaggle/input').rglob('checkpoint_final.pth'):
    ckpt_path = p; break
for fname in ['plans.json', 'nnUNetPlans.json']:  # handles both naming conventions
    for p in Path('/kaggle/input').rglob(fname):
        plans_path = p; break
    if plans_path: break

print(f'Checkpoint: {ckpt_path}')
print(f'Plans:      {plans_path}')

model = None

def safe_torch_load(path):
    # PyTorch 2.6+ blocks numpy in checkpoints with weights_only=True
    import numpy
    try:
        return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        pass
    try:
        safe = [numpy.core.multiarray.scalar, numpy.dtype, numpy.ndarray]
        with torch.serialization.safe_globals(safe):
            return torch.load(path, map_location='cpu', weights_only=True)
    except Exception:
        pass
    # Final fallback - weights_only=False (checkpoint is our own trusted upload)
    return torch.load(path, map_location='cpu', weights_only=False)

# -- Step 2: Build PlainConvUNet directly from plans.json --
if ckpt_path and plans_path:
    try:
        import json as _j, torch.nn as nn

        plans = _j.load(open(plans_path))
        cfg   = plans['configurations']['3d_fullres']

        # Read exact architecture from plans
        arch_class = cfg.get('UNet_class_name', 'PlainConvUNet')
        n_stages   = len(cfg['conv_kernel_sizes'])
        base_f     = cfg.get('UNet_base_num_features', 32)
        max_f      = cfg.get('unet_max_num_features', 320)
        features   = [min(base_f * (2**i), max_f) for i in range(n_stages)]
        strides    = cfg['pool_op_kernel_sizes']
        kernels    = cfg['conv_kernel_sizes']
        n_enc      = cfg.get('n_conv_per_stage_encoder', [2]*n_stages)
        n_dec      = cfg.get('n_conv_per_stage_decoder', [2]*(n_stages-1))
        print(f'  {arch_class} | {n_stages} stages | features: {features}')
        print(f'  strides: {strides}')

        # Import PlainConvUNet - try multiple paths
        PlainConvUNet = None
        for imp in [
            ('dynamic_network_architectures.architectures.unet', 'PlainConvUNet'),
            ('nnunetv2.architectures.neural_network',            'PlainConvUNet'),
            ('nnunetv2.nets.UNet',                               'PlainConvUNet'),
        ]:
            try:
                mod = __import__(imp[0], fromlist=[imp[1]])
                PlainConvUNet = getattr(mod, imp[1])
                print(f'  Imported from {imp[0]}')
                break
            except Exception:
                continue

        if PlainConvUNet is None:
            raise ImportError('Could not import PlainConvUNet from any known path')

        model = PlainConvUNet(
            input_channels          = 4,         # T1N, T1C, T2W, T2F
            n_stages                = n_stages,
            features_per_stage      = features,
            conv_op                 = nn.Conv3d,
            kernel_sizes            = kernels,
            strides                 = strides,
            n_conv_per_stage        = n_enc,
            num_classes             = 3,          # WT, TC, ET
            n_conv_per_stage_decoder= n_dec,
            conv_bias               = False,
            norm_op                 = nn.InstanceNorm3d,
            norm_op_kwargs          = {'eps': 1e-05, 'affine': True},
            dropout_op              = None,
            dropout_op_kwargs       = None,
            nonlin                  = nn.LeakyReLU,
            nonlin_kwargs           = {'inplace': True},
            deep_supervision        = False,
        )

        # Load pretrained weights
        ckpt   = safe_torch_load(ckpt_path)
        state  = ckpt.get('network_weights', ckpt.get('state_dict', ckpt))
        own    = model.state_dict()
        compat = {k: v for k,v in state.items() if k in own and own[k].shape == v.shape}
        model.load_state_dict({**own, **compat}, strict=False)
        n_params = sum(p.numel() for p in model.parameters())
        print(f'  Pretrained: {len(compat)}/{len(own)} layers loaded | {n_params/1e6:.1f}M params')
        if len(compat) < len(own)//2:
            print('  WARNING: <50% layers matched - weight keys may differ')

    except Exception as e:
        print(f'  PlainConvUNet build failed: {e}')
        print('  Falling back to MONAI DynUNet ...')
        model = None

# -- Step 3: Fallback -- MONAI DynUNet --
if model is None:
    from monai.networks.nets import DynUNet
    kernels  = [[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3],[3,3,3]]
    strides  = [[1,1,1],[2,2,2],[2,2,2],[2,2,2],[2,2,2],[2,2,2]]
    model = DynUNet(
        spatial_dims=3, in_channels=4, out_channels=3,
        kernel_size=kernels, strides=strides,
        upsample_kernel_size=strides[1:],
        norm_name='INSTANCE', deep_supervision=False, res_block=True,
    )
    print(f'  DynUNet: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params (training from scratch)')

model = model.to(device)
print(f'Model on {device}')

In [ ]:
import math, time, gc
from torch.cuda.amp import GradScaler, autocast

CKPT_DIR    = OUTPUT_ROOT / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
BEST_PATH   = CKPT_DIR / 'nnunet_best.pth'
LATEST_PATH = CKPT_DIR / 'nnunet_latest.pth'

# ── Recover checkpoints from previous notebook output (attached as input) ──
if not BEST_PATH.exists() or not LATEST_PATH.exists():
    for src_f in sorted(Path('/kaggle/input').rglob('nnunet_best.pth')):
        if not BEST_PATH.exists():
            import shutil; shutil.copy2(src_f, BEST_PATH)
            print(f'  Recovered BEST checkpoint from {src_f}')
        break
    for src_f in sorted(Path('/kaggle/input').rglob('nnunet_latest.pth')):
        if not LATEST_PATH.exists():
            import shutil; shutil.copy2(src_f, LATEST_PATH)
            print(f'  Recovered LATEST checkpoint from {src_f}')
        break
    if BEST_PATH.exists() and not LATEST_PATH.exists():
        # Create LATEST from BEST so train_model() sees start_ep >= epochs
        import shutil; shutil.copy2(BEST_PATH, LATEST_PATH)
        # Patch epoch to >= epochs so training loop skips
        ckpt = torch.load(LATEST_PATH, map_location='cpu')
        if 'epoch' not in ckpt or ckpt['epoch'] < 30:
            ckpt['epoch'] = 30
            torch.save(ckpt, LATEST_PATH)
        print(f'  ✅ Checkpoints recovered — training WILL BE SKIPPED (epoch set to {ckpt["epoch"]})')
    elif BEST_PATH.exists():
        print(f'  ✅ Checkpoints recovered — training will be skipped')

def get_lr(ep, total, base=1e-4):
    warm = 5
    if ep < warm: return base * (ep+1) / warm
    return base * 0.5 * (1 + math.cos(math.pi * (ep-warm) / max(total-warm, 1)))

def safe_loader_iter(loader):
    it = iter(loader)
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream','worker','corrupt']
    while True:
        try:
            yield next(it)
        except StopIteration:
            return
        except Exception as e:
            if any(k in str(e) for k in SKIP):
                continue   # skip bad file, keep training
            raise

def train_model(model, lr=1e-4, epochs=30, patience=10, val_interval=4):
    start_ep, best_dice, mlog = 0, 0.0, {'dice':[],'per_region':[],'loss':[]}
    if LATEST_PATH.exists():
        lc = torch.load(LATEST_PATH, map_location='cpu')
        model.load_state_dict(lc['model'])
        start_ep  = lc.get('epoch',0) + 1
        best_dice = lc.get('best_dice', 0)
        mlog      = lc.get('metrics', mlog)
        print(f'Resumed from epoch {start_ep-1}, best_dice={best_dice:.4f}')
        if start_ep >= epochs:
            return model, best_dice, mlog

    loss_fn     = DiceLoss(to_onehot_y=False, sigmoid=True, smooth_nr=0, smooth_dr=1e-5)
    optimizer   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scaler      = GradScaler()
    dice_metric = DiceMetric(include_background=True, reduction='mean_batch')
    no_improve  = 0
    t0          = time.time()

    try:
        from monai.data import CacheDataset
        train_ds = CacheDataset(train_dicts, train_transforms, cache_rate=0.05, num_workers=4)
        print('Using CacheDataset (5% RAM cache)')
    except Exception:
        train_ds = Dataset(train_dicts, train_transforms)
        print('Using plain Dataset (CacheDataset unavailable)')
    val_ds = Dataset(val_dicts, val_transforms)

    train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,  num_workers=4, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False, num_workers=4, pin_memory=True)
    print(f'Train: {len(train_loader)} batches | Val: {len(val_loader)} batches')
    print(f'Expected ~20 min/epoch x {epochs} epochs ~ {epochs*20/60:.1f}h total')

    for ep in range(start_ep, epochs):
        model.train()
        cur_lr = get_lr(ep, epochs, lr)
        for pg in optimizer.param_groups: pg['lr'] = cur_lr

        ep_loss, n_ok, n_bad = 0.0, 0, 0
        for batch in safe_loader_iter(train_loader):
            try:
                imgs = batch['image'].to(device)
                lbls = batch['label'].to(device)
                optimizer.zero_grad()
                with autocast():
                    loss = loss_fn(model(imgs), lbls)
                scaler.scale(loss).backward()
                scaler.step(optimizer); scaler.update()
                ep_loss += loss.item(); n_ok += 1
            except Exception:
                n_bad += 1
        avg_loss = ep_loss / max(n_ok, 1)
        bad_str  = f' | skipped {n_bad}' if n_bad else ''
        mlog['loss'].append(avg_loss)

        if (ep+1) % val_interval == 0 or ep == epochs-1:
            model.eval(); dice_metric.reset()
            with torch.no_grad():
                for vb in safe_loader_iter(val_loader):
                    try:
                        vo = sliding_window_inference(vb['image'].to(device), PATCH, 2, model, overlap=0.25)
                        dice_metric((torch.sigmoid(vo)>0.5).float(), vb['label'].to(device))
                    except Exception:
                        pass
            dv = dice_metric.aggregate(); md = dv.mean().item()
            pr = [round(dv[i].item(),4) for i in range(3)]  # WT, TC, ET
            mlog['dice'].append(md); mlog['per_region'].append(pr)
            tag = ' NEW BEST' if md > best_dice else ''
            print(f'Ep {ep:3d} | L={avg_loss:.4f} | Dice={md:.4f} WT={pr[0]:.3f} TC={pr[1]:.3f} ET={pr[2]:.3f} | {(time.time()-t0)/60:.1f}m{tag}{bad_str}')
            if md > best_dice:
                best_dice = md; no_improve = 0
                torch.save({'model': model.state_dict(), 'epoch': ep, 'best_dice': best_dice}, BEST_PATH)
            else:
                no_improve += val_interval
            # Save latest for resuming
            torch.save({'model': model.state_dict(), 'epoch': ep,
                        'best_dice': best_dice, 'metrics': mlog}, LATEST_PATH)
            if no_improve >= patience:
                print(f'Early stopping at epoch {ep} (no improvement for {patience} epochs)')
                break
        else:
            print(f'Ep {ep:3d} | L={avg_loss:.4f} | {(time.time()-t0)/60:.1f}m{bad_str}')

    # Load best
    if BEST_PATH.exists():
        model.load_state_dict(torch.load(BEST_PATH, map_location='cpu')['model'])
        print(f'Loaded BEST checkpoint → Dice {best_dice:.4f}')

    # Copy checkpoints to output root for Kaggle download
    for f in [BEST_PATH, LATEST_PATH]:
        dest = OUTPUT_ROOT / f.name
        if not dest.exists():
            shutil.copy2(f, dest)

    return model, best_dice, mlog

model, best_dice, metrics_log = train_model(model)
print(f'\nFine-tuning complete. Best Dice: {best_dice:.4f}')

In [ ]:
import gc, torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from monai.metrics import DiceMetric
import random
import numpy as np
import math
import time

fig_dir = OUTPUT_ROOT / 'figures'; fig_dir.mkdir(exist_ok=True)
REGION_COLORS = {'WT': ('red','lightcoral'), 'TC': ('green','lightgreen'), 'ET': ('blue','lightskyblue')}

# ── Load BEST checkpoint before visualization ──
if BEST_PATH.exists():
    best_ckpt = torch.load(BEST_PATH, map_location='cpu')
    model.load_state_dict(best_ckpt['model'])
    print(f'  Loaded BEST checkpoint → epoch {best_ckpt["epoch"]} | Dice {best_ckpt["best_dice"]:.4f}')
else:
    print('  BEST checkpoint not found — using current weights')
gc.collect(); torch.cuda.empty_cache()

# ── 3D Voxel-Scatter Visualization ──
def visualize_3d_predictions(model, n_samples=5):
    model.eval()
    vis_dicts = random.sample(val_dicts, min(n_samples, len(val_dicts)))
    print(f'  Visualising {len(vis_dicts)} random val patients:')
    for s in vis_dicts: print(f'    {s["patient_id"]} tp={s["timepoint"]}')
    dice_metric = DiceMetric(include_background=True, reduction='none')
    vis_ds     = Dataset(vis_dicts, val_transforms)
    vis_loader = DataLoader(vis_ds, batch_size=1, shuffle=False, num_workers=0)
    for idx, batch in enumerate(vis_loader):
        try:
            gc.collect(); torch.cuda.empty_cache()
            vi = batch['image'].to(device)
            vl = batch['label'].to(device)
            with torch.no_grad():
                vo = sliding_window_inference(vi, PATCH, 1, model, overlap=0.25)
            pred_bin = (torch.sigmoid(vo) > 0.5).float()
            dice_metric.reset(); dice_metric(pred_bin, vl)
            dv  = dice_metric.aggregate()[0].cpu().numpy()
            pid = batch.get('patient_id', ['?'])[0]
            def fmt(v): return 'NaN(empty)' if np.isnan(v) else f'{v:.3f}'
            dstr = '  '.join(f'{r}={fmt(dv[j])}' for j,r in enumerate(REGIONS))
            mean_d = float(np.nanmean(dv))
            title_str = f'Sample {idx} | {dstr}  Mean={mean_d:.3f}  {pid}'
            pred_np = pred_bin.squeeze(0).cpu().numpy()
            gt_np   = vl.squeeze(0).cpu().numpy()
            del vi, vl, vo, pred_bin; gc.collect(); torch.cuda.empty_cache()
            STEP = 3
            fig = plt.figure(figsize=(18, 7))
            fig.suptitle(title_str, fontsize=10, y=1.01)
            for col, (arr, col_title) in enumerate([(gt_np, 'Ground Truth'), (pred_np, 'nnUNet Prediction')]):
                ax = fig.add_subplot(1, 3, col+1, projection='3d')
                ax.set_title(col_title, fontsize=9)
                for ch, (region, (clr, _)) in enumerate(REGION_COLORS.items()):
                    vox = arr[ch]
                    coords = np.argwhere(vox[::STEP, ::STEP, ::STEP] > 0.5)
                    if len(coords) > 0:
                        ax.scatter(coords[:,2]*STEP, coords[:,1]*STEP, coords[:,0]*STEP,
                                   c=clr, alpha=0.3, s=2, label=region)
                ax.legend(fontsize=7); ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
            # Dice bar chart
            ax3 = fig.add_subplot(1, 3, 3)
            colors_bar = [REGION_COLORS[r][0] for r in REGIONS]
            valid_dv = [dv[j] if not np.isnan(dv[j]) else 0 for j in range(3)]
            ax3.barh(REGIONS, valid_dv, color=colors_bar, edgecolor='black')
            ax3.set_xlim(0, 1); ax3.set_xlabel('Dice Score'); ax3.set_title('Per-Region Dice')
            for j, v in enumerate(valid_dv): ax3.text(v+0.01, j, f'{v:.3f}', va='center', fontsize=9)
            plt.tight_layout()
            out_path = fig_dir / f'val_3d_{idx}_{pid}.png'
            plt.savefig(out_path, dpi=150, bbox_inches='tight')
            plt.close()
            print(f'    Saved: {out_path.name} | {dstr}')
        except Exception as e:
            print(f'    Error on sample {idx}: {e}')
            gc.collect(); torch.cuda.empty_cache()

visualize_3d_predictions(model, n_samples=5)
print(f'Figures saved to {fig_dir}')

In [ ]:
import torch.nn.functional as F
import gc, time, random
import numpy as np

# ═══════════ CNN (nnUNet) Embedding Extraction v2 ═════════════════
#
# IDENTICAL strategy to BraTS Phase 2 (Design Decisions D7/D8):
#   - Hook mid encoder stage (~16×16×16 resolution, C channels)
#   - ROI-crop to WT bounding box + 1-cell padding
#   - Octant spatial pooling: 2×2×2 adaptive avg → 8 × C = 8C dims
#   - Mask-weighted pooling: WT/TC/ET within ROI → 3 × C dims
#   - Volumetric morphology: 9-D (log-volumes + presence + ratios)
#   - Total: 8C + 3C + 9 = 11C + 9 dims
#
# For complete resection (no WT):
#   → Octant + region = zero vectors, vol: has_wt=0, etc.
#   → Tumor absence IS the evolution signal
#
# Additionally saves spatial tokens for Phase M3 TaViT V3
# ═══════════════════════════════════════════════════════════════════

def _is_corrupt_file_error(exc):
    SKIP = ['EOFError','gzip','Compressed file','end-of-stream',
            'corrupt','truncat','LoadImaged','applying transform']
    e = exc
    while e is not None:
        if any(k in (type(e).__name__+' '+str(e)) for k in SKIP): return True
        e = e.__cause__ or e.__context__
    return False

def safe_emb_iter(loader):
    it, n_skip = iter(loader), 0
    while True:
        try: yield next(it)
        except StopIteration:
            if n_skip: print(f'  Skipped {n_skip} corrupt files total')
            return
        except Exception as e:
            if _is_corrupt_file_error(e): n_skip += 1; continue
            raise

def get_wt_bbox(lbl_3ch, min_size=2):
    wt = lbl_3ch[0]
    coords = (wt > 0.5).nonzero(as_tuple=True)
    if len(coords[0]) < min_size:
        return None
    z0, z1 = int(coords[0].min()), int(coords[0].max()) + 1
    y0, y1 = int(coords[1].min()), int(coords[1].max()) + 1
    x0, x1 = int(coords[2].min()), int(coords[2].max()) + 1
    D, H, W = wt.shape
    z0, z1 = max(z0-1, 0), min(z1+1, D)
    y0, y1 = max(y0-1, 0), min(y1+1, H)
    x0, x1 = max(x0-1, 0), min(x1+1, W)
    if z1-z0 < 2: z1 = min(z0+2, D)
    if y1-y0 < 2: y1 = min(y0+2, H)
    if x1-x0 < 2: x1 = min(x0+2, W)
    return (z0, z1, y0, y1, x0, x1)

def extract_embeddings(model):
    model.eval()
    _feats = {}; hooks = []

    # ── Hook the MID encoder stage (16×16×16 resolution) ──
    # nnUNet PlainConvUNet: 6 stages, features [32, 64, 128, 256, 320, 320]
    # Stage 3: 32³ → 16³  (256ch)   ← we hook THIS
    if hasattr(model, 'encoder') and hasattr(model.encoder, 'stages'):
        stages = list(model.encoder.stages)
        n = len(stages)
        target_idx = min(3, n - 1)
        def _hook(m, inp, out):
            feat = out[-1] if isinstance(out, (list, tuple)) else out
            _feats['mid'] = feat.detach()
        hooks.append(stages[target_idx].register_forward_hook(_hook))
        print(f'  Hook: encoder.stages[{target_idx}] ({type(stages[target_idx]).__name__})')
    else:
        target = model.encoder if hasattr(model, 'encoder') else model
        def _hook(m, inp, out):
            feat = out[-1] if isinstance(out, (list, tuple)) else out
            _feats['mid'] = feat.detach()
        hooks.append(target.register_forward_hook(_hook))
        print(f'  Hook fallback: {type(target).__name__}')

    emb_dir = OUTPUT_ROOT / 'embeddings'; emb_dir.mkdir(exist_ok=True)
    embs, ids, tps = [], [], []
    spatial_tokens_list = []
    bboxes_list = []

    # Extract from ALL scans (train + val + test)
    ds     = Dataset(all_dicts, val_transforms)
    loader = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0)
    total  = len(all_dicts)
    n_skip = 0; n_empty = 0
    t_start = time.time()

    print(f'  Extraction v2: {total} scans (ALL — train+val+test)')
    print(f'  ⚠️  Using MODEL PREDICTIONS (not GT) for volumes & ROI')
    print(f'  {"─"*60}')

    with torch.no_grad():
        for idx, batch in enumerate(safe_emb_iter(loader)):
            pid = batch['patient_id'][0]
            tp  = batch['timepoint'][0]
            try:
                img = batch['image'].to(device)
                lbl = batch['label'].to(device)
                img_p = F.interpolate(img, PATCH, mode='trilinear', align_corners=False)
                # GT label loaded but not used for embedding (using predicted mask instead)
                _feats.clear()
                # Get model prediction as mask (instead of GT)
                logits = model(img_p)
                pred_mask = (torch.sigmoid(logits) > 0.5).float()
                # Use predicted mask for volumes and ROI
                lbl_pred = pred_mask

                if 'mid' not in _feats:
                    print(f'  [{idx+1:4d}/{total}] WARN no hook → skip {pid}')
                    n_skip += 1; continue

                feat = _feats['mid']
                C = feat.shape[1]

                if idx == 0:
                    print(f'  Feature map: {tuple(feat.shape)} → C={C}')

                # Compute volumes from PREDICTED mask (not GT)
                wt_vol = float(lbl_pred[0, 0].sum().item())
                tc_vol = float(lbl_pred[0, 1].sum().item())
                et_vol = float(lbl_pred[0, 2].sum().item())

                # Downsample labels to feature map resolution
                lbl_feat = F.adaptive_avg_pool3d(lbl_pred, feat.shape[2:])

                # ── Component 1: Octant Spatial Pooling (8 × C) ──
                feat_crop = None
                bbox = get_wt_bbox(lbl_feat[0], min_size=2)

                if bbox is not None:
                    z0, z1, y0, y1, x0, x1 = bbox
                    feat_crop = feat[:, :, z0:z1, y0:y1, x0:x1]
                    oct_pooled = F.adaptive_avg_pool3d(feat_crop, (2, 2, 2))
                    oct_vec = oct_pooled[0].reshape(C, 8).T.reshape(-1)
                    lbl_crop = lbl_feat[:, :, z0:z1, y0:y1, x0:x1]
                    feat_flat = feat_crop[0].reshape(C, -1)
                    region_vecs = []
                    for ch in range(3):
                        mask = lbl_crop[0, ch].reshape(-1)
                        vol_soft = float(mask.sum().item())
                        if vol_soft > 0.01:
                            rvec = (feat_flat * mask.unsqueeze(0)).sum(1) / mask.sum()
                        else:
                            rvec = torch.zeros(C, device=device)
                        region_vecs.append(rvec)
                    roi_size = f'{z1-z0}×{y1-y0}×{x1-x0}'
                    sp_tok = feat_crop[0].reshape(C, -1).T.cpu().numpy()
                    bbox_entry = list(bbox)
                else:
                    oct_vec = torch.zeros(8 * C, device=device)
                    region_vecs = [torch.zeros(C, device=device) for _ in range(3)]
                    roi_size = 'empty'
                    n_empty += 1
                    sp_tok = np.zeros((1, C), dtype=np.float32)
                    bbox_entry = [0, 0, 0, 0, 0, 0]

                # ── Component 3: Volumetric morphology (9-D) ──
                log_wt = np.log1p(wt_vol)
                log_tc = np.log1p(tc_vol)
                log_et = np.log1p(et_vol)
                has_wt = 1.0 if wt_vol > 10 else 0.0
                has_tc = 1.0 if tc_vol > 10 else 0.0
                has_et = 1.0 if et_vol > 10 else 0.0
                # Clamp ratios to [0, 1] — predicted masks may violate TC⊃ET hierarchy
                tc_wt = min(tc_vol / (wt_vol + 1e-6), 1.0)
                et_wt = min(et_vol / (wt_vol + 1e-6), 1.0)
                et_tc = min(et_vol / (tc_vol + 1e-6), 1.0)
                vol_feat = torch.tensor(
                    [log_wt, log_tc, log_et, has_wt, has_tc, has_et,
                     tc_wt, et_wt, et_tc], dtype=torch.float32)

                # ── Concatenate: [octant(8C) + region(3C) + vol(9)] ──
                emb = torch.cat([oct_vec.cpu()] + [v.cpu() for v in region_vecs]
                                + [vol_feat]).numpy()

                del img, lbl, img_p, feat, lbl_pred
                if feat_crop is not None:
                    del feat_crop
                    feat_crop = None

                spatial_tokens_list.append(sp_tok)
                bboxes_list.append(bbox_entry)
                embs.append(emb); ids.append(pid); tps.append(tp)

                if idx == 0:
                    print(f'  Embedding: octant={8*C} + region={3*C} + vol=9 = {emb.shape[0]}-D')
                    print(f'  {"─"*60}')

                if (idx+1) % 50 == 0 or idx == 0:
                    elapsed   = time.time() - t_start
                    rate      = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total - idx - 1) / max(rate, 1e-6)
                    print(
                        f'  [{idx+1:4d}/{total}] {pid[:30]:<30}'
                        f'  tp={tp}'
                        f'  WT={wt_vol:6.0f}v TC={tc_vol:6.0f}v ET={et_vol:6.0f}v'
                        f'  ROI={roi_size}'
                        f'  | {rate:.1f}/s ETA {remaining/60:.1f}m'
                    )
                elif (idx+1) % 10 == 0:
                    elapsed   = time.time() - t_start
                    rate      = (idx+1) / max(elapsed, 1e-6)
                    remaining = (total - idx - 1) / max(rate, 1e-6)
                    print(f'  [{idx+1:4d}/{total}]  done={len(embs)} | {rate:.1f}/s ETA {remaining/60:.1f}m')

            except Exception as e:
                print(f'  [{idx+1:4d}/{total}] ERROR {pid}: {str(e)[:70]}')
                n_skip += 1; continue

    for hk in hooks:
        try: hk.remove()
        except: pass

    elapsed_total = time.time() - t_start
    print(f'  {"─"*60}')
    print(f'  Done: {len(embs)}/{total} in {elapsed_total/60:.1f} min')
    print(f'  Skipped: {n_skip} | Empty ROI (complete resection): {n_empty}')

    if not embs: raise RuntimeError('No embeddings extracted.')
    arr = np.array(embs)
    D = arr.shape[1]
    out = emb_dir / 'cnn_nnunet_embeddings.npz'
    np.savez_compressed(out, embeddings=arr,
                        patient_ids=np.array(ids), timepoints=np.array(tps))
    print(f'  Saved: {out}  shape={arr.shape} ({arr.nbytes/1e6:.1f} MB)')

    # ── Save spatial tokens ──
    if spatial_tokens_list:
        max_tokens = max(t.shape[0] for t in spatial_tokens_list)
        C_tok = spatial_tokens_list[0].shape[1]
        padded = np.zeros((len(spatial_tokens_list), max_tokens, C_tok), dtype=np.float32)
        n_tokens = np.zeros(len(spatial_tokens_list), dtype=np.int32)
        for j, tok in enumerate(spatial_tokens_list):
            padded[j, :tok.shape[0], :] = tok
            n_tokens[j] = tok.shape[0]
        tok_out = emb_dir / 'cnn_spatial_tokens.npz'
        np.savez_compressed(tok_out,
            spatial_tokens=padded,
            token_counts=n_tokens,
            patient_ids=np.array(ids),
            timepoints=np.array(tps),
            bboxes=np.array(bboxes_list)
        )
        print(f'  Spatial tokens: {tok_out}')
        print(f'    Shape: {padded.shape} ({padded.nbytes/1e6:.1f} MB)')
        print(f'    Max tokens: {max_tokens}  Mean: {n_tokens.mean():.0f}')

    # ── Save tumor_volumes.csv ──
    import pandas as pd
    vol_rows = []
    for j, (pid, tp, emb) in enumerate(zip(ids, tps, embs)):
        v = emb[-9:]
        vol_rows.append({
            'patient_id': pid, 'timepoint': tp,
            'wt_vol': float(np.expm1(v[0])),
            'tc_vol': float(np.expm1(v[1])),
            'et_vol': float(np.expm1(v[2])),
            'has_wt': float(v[3]), 'has_tc': float(v[4]), 'has_et': float(v[5]),
            'tc_wt_ratio': float(v[6]), 'et_wt_ratio': float(v[7]),
            'et_tc_ratio': float(v[8]),
        })
    vol_df = pd.DataFrame(vol_rows)
    csv_out = emb_dir / 'tumor_volumes.csv'
    vol_df.to_csv(csv_out, index=False)
    print(f'  Volumes CSV: {csv_out}  ({len(vol_df)} rows)')
    C_val = D - 9
    n_oct = (C_val * 8) // 11
    n_reg = (C_val * 3) // 11
    print(f'  Architecture: octant={n_oct}D + region={n_reg}D + vol=9D = {D}D')
    return arr

import gc
print('Cleaning VRAM before embedding extraction...')
if 'optimizer' in globals(): del optimizer
if 'scaler'    in globals(): del scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9
    print(f'VRAM free: {free_gb:.1f} GB')
else:
    print('Running on CPU')

print('\nExtracting nnUNet embeddings v2 (ROI-crop + octant + volumetric)...')
embeddings = extract_embeddings(model)

In [ ]:
import random

npz_path = OUTPUT_ROOT / 'embeddings' / 'cnn_nnunet_embeddings.npz'
if npz_path.exists():
    data   = np.load(npz_path)
    ids_arr = data['patient_ids']
    emb_arr = data['embeddings']
    N, D    = emb_arr.shape
    norms   = np.linalg.norm(emb_arr, axis=1)

    # Cosine similarity on random pairs
    emb_norm = emb_arr / (norms[:, None] + 1e-8)
    n_pairs  = min(200, N*(N-1)//2)
    pairs    = random.sample([(i,j) for i in range(N) for j in range(i+1,N)], n_pairs)
    sims     = [float(np.dot(emb_norm[i], emb_norm[j])) for i,j in pairs]
    cos_mean = float(np.mean(sims))
    diverse  = 100.0 * float(np.mean(np.array(sims) < 0.95))
    status   = 'GOOD diversity' if diverse > 20 else 'LOW diversity - check embedding hook'

    print('Embedding Diversity Check')
    print(f'  Scans:    {N}')
    print(f'  Dim:      {D}  (octant + 3×region + 9 volumetric)')
    print(f'  Norm:     [{norms.min():.2f}, {norms.max():.2f}]  mean={norms.mean():.2f}')
    print(f'  Cos sim:  mean={cos_mean:.3f} over {n_pairs} random pairs')
    print(f'  Diverse:  {diverse:.1f}% of pairs have cos < 0.95  [{status}]')

    # Per-timepoint check
    tps = data['timepoints']
    for tp in sorted(set(tps)):
        idx = [i for i,t in enumerate(tps) if t == tp]
        print(f'  Timepoint {tp}: {len(idx)} scans')
else:
    print(f'Embeddings not found at {npz_path}')
    print('  Run Cell 10 (embedding extraction) first')

In [ ]:
import json as _j
summary = {
    'model': MODEL_NAME, 'best_dice': float(best_dice),
    'dataset': 'MU-Glioma-Post',
    'regions': REGIONS,
    'label': 'MU-Glioma: WT=1+2+3(NETC+SNFH+ET), TC=1+3(NETC+ET), ET=3, RC=4(excluded)',
    'train_scans': len(train_dicts), 'val_scans': len(val_dicts),
    'all_scans': len(all_dicts),
    'pretrained_source': 'BraTS 2021 fold_0 (original)',
    'target_brats2021': {'WT': 0.9005, 'TC': 0.8673, 'ET': 0.8509},
}
(OUTPUT_ROOT / 'summary.json').write_text(_j.dumps(summary, indent=2))

print('='*55)
print(f'  nnUNet Fine-Tuning Complete (MU-Glioma)')
print(f'  Best Mean Dice:  {best_dice:.4f}')
print(f'  Regions:         {REGIONS}')
print(f'  Target (BraTS2021): WT=0.900 TC=0.867 ET=0.851')
print(f'  Train scans:     {len(train_dicts)}')
print(f'  All scans (emb): {len(all_dicts)}')
print(f'  Outputs: {OUTPUT_ROOT}')
print('='*55)